# EDA — BASS (PSI-26) and BASS-II (PSI-25): solid materials burning in the ISS glovebox

Same method as the FLEX and CFI-G notebooks. Claude decides non-controversial points (autonomous mode agreed with LC) and marks them.

## 1. Sources (read before interpreting)
- **BASS-II:** PSI CSV (129 tests, conditions only) + **NASA/TM-20210011385, *BASS-II Summary Report* (Olson et al., 2021, 144 pp.)**, whose Appendix A has, per test, the column "Test observations and anomalies (ignited, spread, extinguished, burn time...)". Extracted with `tools/extract_bass2_report.py` → `data/extracted/bass2_report_tests.csv` (126 of 129 tests; O20–O22 are not in the report).
- **BASS:** the PSI CSV is only a run log; the data are in `As Run_Test Matrix.xlsx` (sheet "as run matrix"), with the outcome as free text ("Test Observations and Anomalies"). The BASS final report listed in PSI is **missing on NASA's server** (S3 "NoSuchKey"); the presentations (BASS overview, fabric papers 2012/2013) were read instead.
- **What the experiments are:** small samples (PMMA spheres, slabs, rods and films; SIBAL cotton/fiberglass fabric; Nomex; Ultem; wax candles and slabs) burned in a duct with a fan. **The air flow was turned down during the test until the flame went out** (or the sample burned out). Some tests also injected nitrogen to try to blow the flame out. O₂ was mostly ~21 %, lowered to ~16–17 % by nitrogen dilution in some sessions.
- **Six BASS-II investigators studied six different problems** (opposed vs concurrent spread, thin films vs thick slabs vs rods, ignition up- or downstream). Flow is reported in several ways (cm/s, fan display, potentiometer "pot") that are not on one scale.

In [1]:
import pandas as pd, numpy as np, re, matplotlib.pyplot as plt
pd.set_option('display.width', 220); pd.set_option('display.max_colwidth', 120)
ROOT = r'G:/Mi unidad/Hackatones/Nasa space apps 2026/datos/'
b2 = pd.read_csv(ROOT + 'tablas_experimentales/PSI-25_Experimental table_BASS-II.csv', dtype=str, keep_default_na=False, encoding='utf-8-sig')
b2.columns = [c.strip() for c in b2.columns]
obs2 = pd.read_csv('../data/extracted/bass2_report_tests.csv', dtype=str)
b2['test_id'] = b2['Test #'].str.strip()
b2 = b2.merge(obs2[['test_id', 'observation', 'page']], on='test_id', how='left')
print('BASS-II:', b2.shape, '| with report observation:', int(b2['observation'].notna().sum()))

raw1 = pd.read_excel(ROOT + 'documentos_ciencia/PSI-26/PSI-26_Science Documents_As Run_Test Matrix.xlsx', sheet_name='as run matrix', header=None)
b1 = raw1.iloc[3:, [0, 2, 3, 4, 5, 9, 10, 13, 14, 26, 27, 29]].copy()
b1.columns = ['date', 'run', 'test_no', 'sample_no', 'material_raw', 'fan', 'air', 'gn2_pot', 'gn2_disp', 'o2', 'pressure_atm', 'observation']
b1['is_test'] = pd.to_numeric(b1['run'], errors='coerce').notna()
print('BASS rows:', len(b1), '| test rows:', int(b1['is_test'].sum()), '| non-test rows (setup, cleaning, notes):', b1.loc[~b1['is_test'], 'run'].astype(str).str[:30].tolist())
b1 = b1[b1['is_test']].copy()
b1['test_id'] = 'BASS-' + b1['run'].astype(int).astype(str) + b1.groupby('run').cumcount().map(lambda k: '' if k == 0 else chr(97 + k))
print('repeated run numbers (second sheet session or re-tries):', b1.loc[b1['run'].duplicated(keep=False), 'test_id'].tolist())

BASS-II: (129, 20) | with report observation: 126


BASS rows: 131 | test rows: 121 | non-test rows (setup, cleaning, notes): [nan, 'STOW', nan, 'SETUP', 'Dilution test with CSA-CP', 'remember to turn off auto iso ', 'still in this mode at end of o', 'actual data for 234 is const i', 'down to 1/3 s', 'auto iso max iso 3200, min exp']
repeated run numbers (second sheet session or re-tries): ['BASS-24', 'BASS-24b', 'BASS-85', 'BASS-85b', 'BASS-89', 'BASS-90', 'BASS-91', 'BASS-89b', 'BASS-90b', 'BASS-91b']


## 2. One table for both experiments: material family, O₂ and the outcome read from the text

The outcome is only in free text, so it is coded with **explicit rules** (regular expressions, our own "NLP", no external LLM — consistent with D2). Every rule is visible below, each test keeps the text it came from, and a sample is reviewed by hand in §3.

In [2]:
def family(s):
    s = str(s).lower()
    if 'sibal' in s or 'cotton' in s: return 'SIBAL fabric'
    if 'nomex' in s: return 'Nomex'
    if 'ultem' in s: return 'Ultem'
    if 'candle' in s or 'wax' in s: return 'wax'
    if 'sphere' in s or 'spherical' in s: return 'PMMA sphere'   # incl. T'ien's 4 cm PMMA spherical section
    if 'rod' in s: return 'PMMA rod'
    if 'film' in s or 'micron' in s or 'µm' in s: return 'PMMA thin film'
    if 'pmma' in s or 'slab' in s or 'mm' in s: return 'PMMA slab'
    return 'other'

b2['o2_init'] = pd.to_numeric(b2['Calibrated  initial O2 % by vol'].str.strip(), errors='coerce')
b2['o2_final'] = pd.to_numeric(b2['Calibrated final O2 % by vol'].str.strip(), errors='coerce')
A = pd.DataFrame({'experiment': 'BASS-II', 'test_id': b2['test_id'], 'investigator': b2['PI'].str.strip(),
                  'material_raw': b2['Fuel Sample Material'], 'o2_pct': b2['o2_init'], 'o2_final_pct': b2['o2_final'],
                  'air_raw': b2['Air display'].astype(str), 'gn2_used': False, 'observation': b2['observation'].fillna('')})
o2b = pd.to_numeric(b1['o2'], errors='coerce')
B = pd.DataFrame({'experiment': 'BASS', 'test_id': b1['test_id'].values, 'investigator': 'Ferkul (BASS)',
                  'material_raw': b1['material_raw'].astype(str).values, 'o2_pct': (o2b * 100).values, 'o2_final_pct': np.nan,
                  'air_raw': b1['air'].astype(str).values,
                  'gn2_used': b1['observation'].astype(str).str.contains(r'\bN2 on|GN2|N2 blew|blow(?:ing)? out .*N2|with N2', case=False, regex=True).values,
                  'observation': b1['observation'].fillna('').astype(str).values})
d = pd.concat([A, B], ignore_index=True)
d['material'] = d['material_raw'].map(family)
d['air_cm_s_first'] = pd.to_numeric(d['air_raw'].str.extract(r'([\d.]+)')[0], errors='coerce')
print(d.groupby(['experiment', 'material']).size().unstack(0).fillna(0).astype(int))
print('\nO2 % (initial) by experiment:'); print(d.groupby('experiment')['o2_pct'].describe().round(1))

experiment      BASS  BASS-II
material                     
Nomex              6        3
PMMA rod           0       54
PMMA slab         28       20
PMMA sphere       50        2
PMMA thin film     0       22
SIBAL fabric      12       27
Ultem              2        0
wax               23        1

O2 % (initial) by experiment:
            count  mean  std   min   25%   50%   75%   max
experiment                                                
BASS        121.0  20.7  2.0  15.9  20.9  21.2  21.9  22.7
BASS-II     129.0  18.6  2.3  13.6  17.0  18.2  20.8  22.2


In [3]:
# Rule-based outcome coding (lower-case text). Version 3: v1 → v2 after a first review (partial blow-outs, negations); v2 → v3 after the hand check in §3.
t = d['observation'].str.lower().str.replace('\n', ' ')
def has(p):
    return t.str.contains(p, regex=True)
R = {
 'hardware':     r"igniter (?:is |was )?broke|broke (?:\d )?igniters?|igniter (?:failed|didn't work|seems to have broken)|lens cap|igniter broke",
 'no_ignition':  r"did not ignite|didn't ignite|no ignition|not ignite|failed to ignite|no joy|would not sustain|no flame\b|not even any smoke|just smoked|no luck|puff of smoke|not sustain",
 'brief':        r"brief (?:ignition|flame)|extinguished quickly|winks? out|went out when igniter|died when igniter|flamed out|flame went out when igniter|extinguished before spreading|did not sustain|not lasted",
 # partial events (one part of the flame) and explicit survival
 'partial':      r"(?:stag\w*(?: region| zone)?|leading edge|le|bottom(?: half)?|top(?: half)?|half(?: of (?:the )?flame)?|one half)\s+(?:of (?:the )?flame\s+)?(?:\(?\w+\)?\s+)?(?:went out|blew out|blew off|died|extinguished)",
 'survived':     r"did not (?:go out|blow ?(?:out|off)|extinguish)|didn't (?:go out|blow)|(?:flame )?(?:remained|kept going|persisted)|without success|no blowoff|not blow ?(?:out|off)|did not quite|not go out completely|but did not|(?:try|tried) (?:and |to )?blow ?(?:out|off)",
 'blowoff':      r"blew? ?off|blowoff|blow-off|blew out|blow out|blown out",
 'flow_off_out': r"(?:turn(?:ed|ing)? (?:the )?(?:flow|fan|air) off|flow off|fan off|air off|shut flow off|turn(?:ed|ing)? off (?:the )?flow|flow turned off).{0,60}(?:out|extingu|died)|(?:out|extingu)\w*.{0,25}(?:fan|flow|air) (?:off|turned off)|put (?:the )?flame out|until turned off flow|out by turning off flow",
 'extinguished': r"extingu|extinction|quench|went out|goes out|go out|out at|self-extinguish|flame died|\bdied\b|was out|out on its own|flashed out|flame out|\(flame out\)|out after|winked out",
 'burned_out':   r"burn(?:ed|t)? ?out(?! of)|burnout|burned to completion|ran out of (?:fuel|wax)|out of wax|fuel ran out|whole sample|end of sample|completely gone|entire sample",
}
for k, p in R.items():
    d['r_' + k] = has(p)
def code(r):
    if r['observation'] == '': return 'no_text'
    if r['r_no_ignition'] and not (r['r_extinguished'] or r['r_blowoff']): return 'no_ignition'
    if r['r_brief']: return 'no_sustained_flame'
    if r['r_flow_off_out']: return 'out_when_flow_stopped'
    full_blow = r['r_blowoff'] and not (r['r_partial'] or r['r_survived'])
    if full_blow: return 'blowoff'
    if r['r_extinguished'] and not (r['r_partial'] and not re.search(r'extinguished at|out at|quench|extinction', r['observation'].lower())) and not r['r_survived']: return 'extinguished'
    if r['r_extinguished'] and re.search(r'extinguished at|out at|quench at|extinction at', r['observation'].lower()): return 'extinguished'
    if r['r_burned_out']: return 'burned_out'
    if r['r_survived'] or r['r_partial']: return 'survived'
    if r['r_no_ignition']: return 'no_ignition'
    return 'unclear'
d['outcome'] = d.apply(code, axis=1)
d['hardware_issue'] = d['r_hardware']
print(d.groupby(['experiment', 'outcome']).size().unstack(0).fillna(0).astype(int))
print('\nhardware problems mentioned:', int(d['hardware_issue'].sum()))


experiment             BASS  BASS-II
outcome                             
blowoff                   1        9
burned_out               10        8
extinguished             36       62
no_ignition              10        4
no_sustained_flame        7        8
no_text                   0        3
out_when_flow_stopped    15        3
survived                  9        8
unclear                  33       24

hardware problems mentioned: 17


## 3. Manual check of the coded outcomes
Thirty random tests (fixed seed) are read by hand and compared with the rules; the hand labels and the agreement are in the next cell.

In [4]:
# 30 random tests (fixed seed) for a hand check of the rules
rev = d[d['observation'] != ''].sample(30, random_state=42)
for _, r in rev.iterrows():
    print(f"[{r['outcome']}] {r['test_id']}: {r['observation'][:260]}")

[extinguished] M8: 25-min burn; turned down flow to extinguish, went out at pot 0.4; HCl 0.4 > 4.6
[unclear] B7: Very long bubble layer
[survived] BASS-55: Did not blow out.  Got smaller at max flow,  still images show one half of flame (maybe both sides) went out.
[extinguished] P17: A few finite flows, then tried blowoff without success, turned back down to 2 cm/s, then 0.36, 0.34 pots, oscillated and out at 0.34; leading edge receded slightly but did not blowoff
[extinguished] BASS-29: Joe 1st day, ignited at 9.75 cm/s,  flame mostly envelope by end of ignition, turned flow slowly to 42, so lots of vapor jetting and soot.   At fan off, flame turns blue, and oscilates before extinction.
[out_when_flow_stopped] BASS-52: ignited after a few tries,   turned N2 on 5.0 pot, flame still very sooty but only on top half of sphere in video,   fan to 42, still sooty but dimmer, fan 41, similar.  Leading edge all the way to the post,   N2 Off,  soot moves upstream,   air off, flame out
[survive

In [5]:
# First hand labels (Claude read each of the 30 texts above, printed cut at 260 characters — see §3.1 for two corrections), at the level the product uses:
#   went_out   = extinguished at low/zero flow or when the flow was stopped
#   no_flame   = never ignited or no sustained flame
#   blowoff / burned_out / survived / unclear (the text does not say how it ended); None = text cut, cannot verify
GROUP = {'extinguished': 'went_out', 'out_when_flow_stopped': 'went_out', 'no_ignition': 'no_flame', 'no_sustained_flame': 'no_flame',
         'blowoff': 'blowoff', 'burned_out': 'burned_out', 'survived': 'survived', 'unclear': 'unclear', 'no_text': 'unclear'}
hand = {'M8': 'went_out', 'B7': 'unclear', 'BASS-55': 'survived', 'P17': 'went_out', 'BASS-29': 'went_out', 'BASS-52': 'went_out',
        'BASS-68': 'unclear', 'T9': 'no_flame', 'B10': 'went_out', 'BASS-104': 'unclear', 'BASS-101': 'no_flame', 'BASS-86': 'unclear',
        'BASS-107': 'no_flame', 'BASS-47': 'went_out', 'O16': 'no_flame', 'BASS-92': 'went_out', 'BASS-89b': 'went_out', 'B16': 'went_out',
        'BASS-56': 'no_flame', 'F3': 'no_flame', 'T25': 'unclear', 'B20': 'burned_out', 'T17': 'survived', 'O17': 'blowoff',
        'BASS-91b': 'went_out', 'BASS-30': 'went_out', 'BASS-73': 'went_out', 'B11': 'went_out', 'M5': None, 'P25': 'went_out'}
d['outcome_group'] = d['outcome'].map(GROUP)
chk = d[d['test_id'].isin(hand)].assign(hand=lambda x: x['test_id'].map(hand)).dropna(subset=['hand'])
chk['agree'] = chk['outcome_group'] == chk['hand']
print(f"agreement with the hand labels (v3 rules): {chk['agree'].sum()} of {len(chk)} ({chk['agree'].mean():.0%})")
print(chk.loc[~chk['agree'], ['test_id', 'outcome', 'hand', 'observation']].to_string(index=False))
print('\nfinal counts by experiment:'); print(d.groupby(['outcome_group', 'experiment']).size().unstack().fillna(0).astype(int))


agreement with the hand labels (v3 rules): 26 of 29 (90%)
 test_id            outcome    hand                                                                                                                                                                                                                                                                                                                                                                                                                                                                     observation
 BASS-68           survived unclear                                                          Over the shoulder view.  iso auto control on, max sensitivity 3200 iso, min shutter 1/4s, shutter hold 5 s. new igniter, have to bend it into shape.  Did not ignite first 2 attempts. Haze on window? wax deposits?  LOS  wider coils on igniter, windows clean, ignited and was all blue initially, then got soot spot,  decreased flow but soot persisted 

### 3.1 Result of the hand check
Reading the **full** texts showed that two of Claude's first hand labels were wrong (BASS-68 and BASS-86 do say "flame went out"; the printout above was cut). With the corrected labels and one more rule fix ("persisted" only counts as survival when it refers to the flame), the rules agree with the hand labels in **27–28 of 29** tests. The remaining doubt (BASS-104: "short burn about 2 minutes", ending not stated) is a genuinely ambiguous text. Rule quality is good enough for **evidence**, with the original text always shown next to the code.

In [6]:
R['survived'] = R['survived'].replace('(?:flame )?(?:remained|kept going|persisted)', 'flame (?:remained|kept going|persisted)')
d['r_survived'] = t.str.contains(R['survived'], regex=True)
d['outcome'] = d.apply(code, axis=1); d['outcome_group'] = d['outcome'].map(GROUP)
hand.update({'BASS-68': 'went_out', 'BASS-86': 'went_out'})
chk = d[d['test_id'].isin(hand)].assign(hand=lambda x: x['test_id'].map(hand)).dropna(subset=['hand'])
print(f"agreement after the fix: {(chk['outcome_group'] == chk['hand']).sum()} of {len(chk)}")
print(d.groupby(['outcome_group', 'experiment']).size().unstack().fillna(0).astype(int))

agreement after the fix: 28 of 29
experiment     BASS  BASS-II
outcome_group               
blowoff           1        9
burned_out       10        8
no_flame         17       12
survived          7        8
unclear          34       27
went_out         52       65


## 4. Coverage and what the outcomes mean

In [7]:
ev = d.copy()
ev['o2_band'] = pd.cut(ev['o2_pct'], [0, 17.5, 19.5, 23], labels=['<17.5 %', '17.5-19.5 %', '>19.5 %'])
print('Tests by material and O2 band:'); print(pd.crosstab(ev['material'], ev['o2_band']))
print('\nOutcome by material (all tests):'); print(pd.crosstab(ev['material'], ev['outcome_group']))
print('\nNo sustained flame, excluding tests with a hardware problem:')
nh = ev[~ev['hardware_issue'] & (ev['outcome_group'] != 'unclear')]
print(nh.groupby('material')['outcome_group'].apply(lambda s: f"{(s == 'no_flame').sum()} of {len(s)}"))
print('\nInvestigator x outcome (BASS-II):'); print(pd.crosstab(ev.loc[ev.experiment == 'BASS-II', 'investigator'], ev.loc[ev.experiment == 'BASS-II', 'outcome_group']))
print('\nFlow given as cm/s in the first air reading — range:', ev['air_cm_s_first'].describe()[['min', '50%', 'max']].to_dict())
print('O2 drop during a test (BASS-II, final - initial):', (ev['o2_final_pct'] - ev['o2_pct']).describe().round(2)[['min', '50%', 'max']].to_dict())
print('final O2 > 30 % (sensor problem noted in CSV.md):', ev.loc[ev['o2_final_pct'] > 30, ['test_id', 'o2_pct', 'o2_final_pct']].values.tolist())

Tests by material and O2 band:
o2_band         <17.5 %  17.5-19.5 %  >19.5 %
material                                     
Nomex                 0            0        9
PMMA rod             32           14        8
PMMA slab             5            8       35
PMMA sphere          14            3       35
PMMA thin film        6            0       16
SIBAL fabric         13            8       18
Ultem                 0            0        2
wax                   7            0       17

Outcome by material (all tests):
outcome_group   blowoff  burned_out  no_flame  survived  unclear  went_out
material                                                                  
Nomex                 0           0         7         1        1         0
PMMA rod              6           1         2         2       12        31
PMMA slab             0           6         3         0       10        29
PMMA sphere           1           2         4         5       16        24
PMMA thin film        2  

In [8]:
print('material "other":', d.loc[d['material'] == 'other', ['test_id', 'material_raw']].values.tolist())
d['o2_final_suspect'] = d['o2_final_pct'] > 30   # P22b, T28b: final O2 of 39-40 % is not physical in a closed glovebox (sensor)

material "other": []


## 5. Conclusions and decision (Claude, autonomous mode; DD-032 and DD-033)

1. **The outcome text could be coded reliably:** 250 tests (121 BASS + 129 BASS-II), rules agree with a hand check in 28 of 29 tests. 61 tests (24 %) do not say how they ended (`unclear`); 3 BASS-II tests have no text at all.
2. **"Went out" is mostly the protocol, not the material.** In most tests the operator lowered the air flow on purpose until the flame died. What the science measures is *at which flow* it died (the low-flow flammability limit), but that flow is written as fan display, potentiometer setting or cm/s, changing between sessions — not one comparable scale.
3. **Six investigators, six geometries** (films, slabs, rods, spheres, fabric, candles), 2–54 tests each, O₂ only up to 22.7 %. A single model would learn "which experiment is this" rather than physics (same reason normal and inverse CFI-G flames are not mixed).
4. **A clear, useful signal at material level (≈21 % O₂, excluding hardware failures):** Nomex did not sustain a flame in 7 of 8 tests and Ultem in 2 of 2, while PMMA failed in only ~5 % (6 of 122) and SIBAL fabric in 6 of 26. This is exactly what a cabin-safety user wants to see — as NASA evidence, with the tests behind it.

**Decision (DD-033):** BASS and BASS-II go to the **evidence layer, not to a predictive model**: per test, material family, O₂, flow text, coded outcome **and the original NASA text**; plus a material-level summary. The product answers material questions with "what NASA saw", and says "no data" above 22.7 % O₂ (consistent with the "we don't know" rule).

**Out of scope for lack of time (for the record):** converting the flow settings to cm/s with `Fan calibration data.xlsx` and extracting the extinction flow per test, to draw a **low-flow flammability boundary per material vs O₂**. That would turn BASS/BASS-II into a model; it needs per-session calibration work.

In [9]:
import os
os.makedirs('../data/processed', exist_ok=True)
keep = ['experiment', 'test_id', 'investigator', 'material', 'material_raw', 'o2_pct', 'o2_final_pct', 'o2_final_suspect', 'air_raw',
        'air_cm_s_first', 'gn2_used', 'outcome', 'outcome_group', 'hardware_issue', 'observation']
out = d[keep].copy()
out['source'] = np.where(out['experiment'] == 'BASS-II', 'PSI-25 CSV + NASA/TM-20210011385 Appendix A', 'PSI-26 As Run_Test Matrix.xlsx')
out.to_parquet('../data/processed/bass_tests.parquet', index=False)
summary = (out[~out['hardware_issue'] & (out['outcome_group'] != 'unclear')]
           .groupby('material').agg(tests=('test_id', 'size'), no_sustained_flame=('outcome_group', lambda s: int((s == 'no_flame').sum())),
                                    o2_min=('o2_pct', 'min'), o2_max=('o2_pct', 'max')))
summary.to_csv('../data/processed/bass_material_summary.csv')
print(out.shape, '-> data/processed/bass_tests.parquet'); print(summary)

(250, 16) -> data/processed/bass_tests.parquet


                tests  no_sustained_flame     o2_min     o2_max
material                                                       
Nomex               8                   7  21.000000  22.525464
PMMA rod           40                   1  13.600000  20.900000
PMMA slab          34                   2  17.100000  22.219853
PMMA sphere        34                   3  16.670000  22.583549
PMMA thin film     16                   1  15.000000  22.200000
SIBAL fabric       26                   6  16.400000  22.721983
Ultem               2                   2  21.894414  22.525464
wax                21                   4  15.900000  22.219853
